# Equatorial Pacific Thermocline Tilt

The equatorial Pacific thermocline slopes from ~150–200 m in the western
warm pool to ~30–50 m in the eastern cold tongue. This tilt is maintained
by trade winds and is a fundamental test of coupled model fidelity. This
diagnostic computes the depth of the 20°C isotherm (D20) and the depth
of maximum dT/dz across the tropical Pacific and compares against WOA13
observations.

Authors: John Krasting

In [ ]:
# Development mode: constantly refreshes module code
%load_ext autoreload
%autoreload 2

## Framework Code and Diagnostic Setup

In [ ]:
import esnb
from esnb import NotebookDiagnostic, RequestedVariable, CaseGroup2, nbtools
from esnb.sites.gfdl import call_dmget, convert_to_momgrid
import climplot

In [ ]:
# Define requested variables: monthly 3D temperature on z-space grid
variables = [
    RequestedVariable("thetao", "ocean_monthly_z", frequency="mon"),
]

# Diagnostic configuration
diag_name = "Equatorial Pacific Thermocline Tilt"
diag_desc = "Equatorial Pacific thermocline tilt analysis: D20, max dT/dz, and WOA13 comparison"
user_options = {}

# Initialize the diagnostic
diag = NotebookDiagnostic(diag_name, diag_desc, variables=variables, **user_options)

In [ ]:
# Define experiments to analyze
groups = [
    CaseGroup2("am5dt-517", date_range=("0001-01-01", "0040-12-31"),
               name="AM5-DT piControl", plot_color="royalblue"),
    CaseGroup2("odiv-1", date_range=("0001-01-01", "0040-12-31"),
               name="CM4 piControl", plot_color="darkorange"),
    CaseGroup2("cm5-11", date_range=("0001-01-01", "0040-12-31"),
               name="CM5 piControl", plot_color="seagreen"),
]

In [ ]:
# Combine experiments with the diagnostic request and determine files to load
diag.resolve(groups)

In [ ]:
# Print a list of file paths
_ = [print(x) for x in diag.files]

*(The files above are necessary to run the diagnostic.)*

In [ ]:
# Stage files from mass storage
call_dmget(diag.files, status=True)
call_dmget(diag.files)

In [ ]:
# Load the data as xarray datasets
diag.open()

In [ ]:
convert_to_momgrid(diag, return_corners=True)

## Main Diagnostic

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
from momgrid.geoslice import geoslice
import momgrid.util
from momgrid.VerticalSplitScale import VerticalSplitScale

In [ ]:
all_figs = []
climplot.publication()

In [ ]:
def isotherm_depth(temp, z, iso_temp=20.0):
    """Find depth of a given isotherm via linear interpolation.

    Parameters
    ----------
    temp : xarray.DataArray
        Temperature array with a depth dimension (z_l).
    z : numpy.ndarray
        1-D depth coordinate values (positive down, in metres).
    iso_temp : float
        Isotherm temperature to find (default 20 degC).

    Returns
    -------
    d_iso : numpy.ndarray
        Depth of the isotherm (metres). NaN where the isotherm does not exist.
    """
    temp_vals = temp.values if hasattr(temp, "values") else np.asarray(temp)

    shape_hz = temp_vals.shape[1:]  # everything after z dimension
    d_iso = np.full(shape_hz, np.nan)

    for k in range(len(z) - 1):
        t_upper = temp_vals[k]
        t_lower = temp_vals[k + 1]

        # Where iso_temp lies between these two levels
        crosses = ((t_upper >= iso_temp) & (t_lower < iso_temp)) | \
                  ((t_upper <= iso_temp) & (t_lower > iso_temp))
        # Only fill where not already found (take shallowest crossing)
        fill = crosses & np.isnan(d_iso)

        if not fill.any():
            continue

        # Linear interpolation
        frac = (iso_temp - t_upper[fill]) / (t_lower[fill] - t_upper[fill])
        d_iso[fill] = z[k] + frac * (z[k + 1] - z[k])

    return d_iso


def max_gradient_depth(temp, z):
    """Find depth of maximum |dT/dz|.

    Parameters
    ----------
    temp : xarray.DataArray
        Temperature array with a depth dimension (z_l).
    z : numpy.ndarray
        1-D depth coordinate values (positive down, in metres).

    Returns
    -------
    d_max : numpy.ndarray
        Depth of maximum vertical temperature gradient. NaN where all-NaN.
    """
    temp_vals = temp.values if hasattr(temp, "values") else np.asarray(temp)

    nz = len(z)
    dz = np.diff(z)
    z_mid = 0.5 * (z[:-1] + z[1:])

    # Compute |dT/dz| at midpoints
    shape_hz = temp_vals.shape[1:]
    grad = np.full((nz - 1,) + shape_hz, np.nan)
    for k in range(nz - 1):
        grad[k] = np.abs(temp_vals[k + 1] - temp_vals[k]) / dz[k]

    # Find depth of maximum gradient — handle all-NaN columns (land)
    d_max = np.full(shape_hz, np.nan)
    valid = np.any(np.isfinite(grad), axis=0)

    if not valid.any():
        return d_max

    # Replace NaN with -inf before argmax so all-NaN columns give 0 index
    # but we only use the result where valid=True
    grad_filled = np.where(np.isfinite(grad), grad, -np.inf)
    max_idx = np.argmax(grad_filled, axis=0)

    if valid.ndim == 2:
        rows, cols = np.where(valid)
        d_max[rows, cols] = z_mid[max_idx[rows, cols]]
    elif valid.ndim == 1:
        idx = np.where(valid)[0]
        d_max[idx] = z_mid[max_idx[idx]]
    elif valid.ndim == 0 and valid:
        d_max = z_mid[max_idx]

    return d_max

In [ ]:
WOA13_OBS_PATHS = {
    "om4": "/archive/jpk/datasets/OM5/obs/WOA13/WOA13_z_35level_OM4_1080x1440_annual_v20250214.nc",
    "om5": "/archive/jpk/datasets/OM5/obs/WOA13/WOA13_z_35level_OM5_1161x1440_annual_v20240602.nc",
}

_woa13_cache = {}

def load_woa13(obs_key):
    """Load and cache WOA13 observations for the appropriate grid."""
    global _woa13_cache
    if obs_key not in _woa13_cache:
        import momgrid
        ds = xr.open_dataset(WOA13_OBS_PATHS[obs_key])
        ds = momgrid.util.reset_nominal_coords(ds)
        # Fix z_l if it contains level indices instead of depths (OM5 file issue)
        if ds.z_l.values[0] == 0 and ds.z_l.values[1] == 1:
            ds_ref = xr.open_dataset(WOA13_OBS_PATHS["om4"])
            ds = ds.assign_coords(z_l=ds_ref.z_l)
            ds_ref.close()
        _woa13_cache[obs_key] = ds
    return _woa13_cache[obs_key]


def get_obs_key(model_type):
    """Determine obs grid key from model type string."""
    if "esm4" in model_type:
        return "esm4"
    if "om4" in model_type:
        return "om4"
    return "om5"

In [ ]:
# Configuration
ISO_TEMP = 20.0  # isotherm temperature (degC)
EQ_BAND = 5.0    # equatorial band half-width (degrees)
TROP_BAND = 25.0 # tropical band half-width for maps (degrees)
WEST_BOX = (150.0, 180.0)   # western Pacific longitude range (degE, 0-360)
EAST_BOX = (230.0, 280.0)   # eastern Pacific longitude range (degE, 0-360)
MAX_DEPTH = 500.0            # upper ocean depth limit (m)

thetao_var = diag.varmap["thetao"]
results = {}

def process_experiment(group):
    """Load data, compute D20 and max dT/dz for one experiment."""
    ds = group.datasets[thetao_var]
    model_type = ds.attrs.get("model", "")
    obs_key = get_obs_key(model_type)

    # Lazy subset: upper ocean + tropical Pacific (still dask-backed)
    # Use negative lons matching OM4 geolon convention (-300 to 60)
    ds_sub = geoslice(ds.sel(z_l=slice(0, MAX_DEPTH)),
                      x=(-260, -60), y=(-TROP_BAND, TROP_BAND))

    # Convert geolon from model convention to 0-360 for plotting and masking
    ds_sub = ds_sub.assign_coords(geolon=ds_sub.geolon % 360)
    if "geolon_c" in ds_sub.coords:
        ds_sub = ds_sub.assign_coords(geolon_c=ds_sub.geolon_c % 360)

    # Compute annual mean lazily with dask, THEN load (much smaller)
    thetao_annual = ds_sub.thetao.groupby("time.year").mean("time").load()

    # Time-mean over all years
    thetao_clim = thetao_annual.mean("year")

    # Depth coordinate
    z_l = ds_sub.z_l.values

    # Compute D20 (time-mean climatology)
    d20 = isotherm_depth(thetao_clim, z_l, iso_temp=ISO_TEMP)

    # Compute max dT/dz depth
    d_maxgrad = max_gradient_depth(thetao_clim, z_l)

    # D20 for each year (for time evolution)
    d20_annual = np.zeros((len(thetao_annual.year),) + thetao_clim.shape[1:])
    for yi, yr in enumerate(thetao_annual.year.values):
        d20_annual[yi] = isotherm_depth(thetao_annual.sel(year=yr), z_l, iso_temp=ISO_TEMP)
    d20_annual = xr.DataArray(
        d20_annual,
        dims=("year", "yh", "xh"),
        coords={"year": thetao_annual.year,
                "yh": ds_sub.yh, "xh": ds_sub.xh},
    )

    # Equatorial average (within +/- EQ_BAND degrees)
    geolat_1d = ds_sub.geolat.mean("xh") if "geolat" in ds_sub else ds_sub.yh
    eq_mask = np.abs(geolat_1d) <= EQ_BAND
    eq_indices = np.where(eq_mask.values if hasattr(eq_mask, "values") else eq_mask)[0]

    # Equatorial cross-section of temperature
    thetao_eq = thetao_clim.isel(yh=eq_indices).mean("yh")

    # Equatorial D20 profile (longitude)
    d20_eq = np.nanmean(d20[eq_indices, :], axis=0)
    d_maxgrad_eq = np.nanmean(d_maxgrad[eq_indices, :], axis=0)

    # Equatorial D20 timeseries per year
    d20_eq_annual = d20_annual.isel(yh=eq_indices).mean("yh")

    # Get longitude coordinate (now in 0-360 after conversion above)
    geolon_1d = ds_sub.geolon.mean("yh") if "geolon" in ds_sub else ds_sub.xh

    # Corner coords and areacello for map plots
    geolon_c = ds_sub.geolon_c.values if "geolon_c" in ds_sub.coords else None
    geolat_c = ds_sub.geolat_c.values if "geolat_c" in ds_sub.coords else None
    areacello = ds_sub.areacello if "areacello" in ds_sub else None

    return {
        "thetao_clim": thetao_clim,
        "thetao_eq": thetao_eq,
        "d20_map": d20,
        "d20_eq": d20_eq,
        "d_maxgrad_eq": d_maxgrad_eq,
        "d20_annual": d20_annual,
        "d20_eq_annual": d20_eq_annual,
        "z_l": z_l,
        "geolon_1d": geolon_1d,
        "geolat_1d": geolat_1d if hasattr(geolat_1d, "values") else geolat_1d,
        "geolon_c": geolon_c,
        "geolat_c": geolat_c,
        "areacello": areacello,
        "eq_indices": eq_indices,
        "obs_key": obs_key,
        "model_type": model_type,
        "ds_sub": ds_sub,
        "yh_sub": ds_sub.yh,
        "xh_sub": ds_sub.xh,
    }

In [ ]:
# Process experiment 1
group = diag.groups[0]
print(f"Processing {group.name}...")
results[group.name] = process_experiment(group)
print(f"  Done: {group.name}")

In [ ]:
# Process experiment 2
group = diag.groups[1]
print(f"Processing {group.name}...")
results[group.name] = process_experiment(group)
print(f"  Done: {group.name}")

In [ ]:
# Process experiment 3
group = diag.groups[2]
print(f"Processing {group.name}...")
results[group.name] = process_experiment(group)
print(f"  Done: {group.name}")

print(f"\nAll {len(diag.groups)} experiments processed.")

In [ ]:
# Compute D20 from WOA13 observations for each grid type encountered
obs_results = {}

for group in diag.groups:
    r = results[group.name]
    obs_key = r["obs_key"]
    if obs_key in obs_results:
        continue

    dsobs = load_woa13(obs_key)

    # Subset WOA13 to match model subset using yh/xh index ranges
    yh_min, yh_max = float(r["yh_sub"].min()), float(r["yh_sub"].max())
    xh_min, xh_max = float(r["xh_sub"].min()), float(r["xh_sub"].max())
    dsobs_sub = dsobs.sel(
        z_l=dsobs.z_l[dsobs.z_l <= MAX_DEPTH],
        yh=slice(yh_min, yh_max),
        xh=slice(xh_min, xh_max),
    )
    dsobs_sub = dsobs_sub.load()

    # Add geographic coords from model data for consistent plotting
    ds_model = r["ds_sub"]
    if "geolon" in ds_model.coords:
        # Trim to matching shape if needed
        ny = min(dsobs_sub.sizes["yh"], ds_model.sizes["yh"])
        nx = min(dsobs_sub.sizes["xh"], ds_model.sizes["xh"])
        dsobs_sub = dsobs_sub.isel(yh=slice(0, ny), xh=slice(0, nx))
        dsobs_sub = dsobs_sub.assign_coords(
            geolon=ds_model.geolon.isel(yh=slice(0, ny), xh=slice(0, nx)),
            geolat=ds_model.geolat.isel(yh=slice(0, ny), xh=slice(0, nx)),
        )

    thetao_obs = dsobs_sub.thetao.squeeze()
    z_l_obs = dsobs_sub.z_l.values

    d20_obs = isotherm_depth(thetao_obs, z_l_obs, iso_temp=ISO_TEMP)
    d_maxgrad_obs = max_gradient_depth(thetao_obs, z_l_obs)

    # Equatorial averages — use same indices as model (clamp to obs shape)
    eq_idx_obs = r["eq_indices"]
    eq_idx_obs = eq_idx_obs[eq_idx_obs < dsobs_sub.sizes["yh"]]

    thetao_obs_eq = thetao_obs.isel(yh=eq_idx_obs).mean("yh")
    d20_obs_eq = np.nanmean(d20_obs[eq_idx_obs, :], axis=0)
    d_maxgrad_obs_eq = np.nanmean(d_maxgrad_obs[eq_idx_obs, :], axis=0)

    geolon_obs_1d = dsobs_sub.geolon.mean("yh") if "geolon" in dsobs_sub else dsobs_sub.xh

    obs_results[obs_key] = {
        "thetao_eq": thetao_obs_eq,
        "d20_map": d20_obs,
        "d20_eq": d20_obs_eq,
        "d_maxgrad_eq": d_maxgrad_obs_eq,
        "z_l": z_l_obs,
        "geolon_1d": geolon_obs_1d,
        "geolat_1d": r["geolat_1d"],
        "eq_indices": eq_idx_obs,
        "dsobs_sub": dsobs_sub,
    }

print("WOA13 D20 computed for:", list(obs_results.keys()))

In [ ]:
# Equatorial longitude-depth cross-sections of temperature with D20 contour
cmap_t, norm_t, levels_t = climplot.sequential_cmap(0, 30, 1, cmap_name="RdYlBu_r")

for group in diag.groups:
    r = results[group.name]
    obs_key = r["obs_key"]
    obs = obs_results[obs_key]

    lon = r["geolon_1d"].values if hasattr(r["geolon_1d"], "values") else r["geolon_1d"]
    lon_obs = obs["geolon_1d"].values if hasattr(obs["geolon_1d"], "values") else obs["geolon_1d"]
    z = r["z_l"]
    z_obs = obs["z_l"]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

    # Model
    ax = axes[0]
    lon_i = momgrid.util.infer_bounds(lon)
    z_i = momgrid.util.infer_bounds(z, start=0.0)
    cb = ax.pcolormesh(lon_i, z_i, r["thetao_eq"].values, cmap=cmap_t, norm=norm_t)
    ax.contour(lon, z, r["thetao_eq"].values, levels=[ISO_TEMP],
               colors="k", linewidths=1.5)
    ax.invert_yaxis()
    ax.set_ylim(MAX_DEPTH, 0)
    ax.set_ylabel("Depth (m)")
    ax.set_xlabel("Longitude")
    ax.set_title(f"{group.name}")

    # WOA13
    ax = axes[1]
    lon_obs_i = momgrid.util.infer_bounds(lon_obs)
    z_obs_i = momgrid.util.infer_bounds(z_obs, start=0.0)
    cb = ax.pcolormesh(lon_obs_i, z_obs_i, obs["thetao_eq"].values,
                       cmap=cmap_t, norm=norm_t)
    ax.contour(lon_obs, z_obs, obs["thetao_eq"].values, levels=[ISO_TEMP],
               colors="k", linewidths=1.5)
    ax.invert_yaxis()
    ax.set_ylim(MAX_DEPTH, 0)
    ax.set_xlabel("Longitude")
    ax.set_title("WOA13")

    plt.subplots_adjust(bottom=0.18)
    cax = fig.add_axes([0.15, 0.05, 0.7, 0.025])
    fig.colorbar(cb, cax=cax, orientation="horizontal", extend="both",
                 label="Temperature (°C)")
    fig.suptitle(f"Equatorial Cross-Section (±{EQ_BAND}°N/S) — {group.name} vs WOA13",
                 y=1.01)
    all_figs.append(fig)

### Analysis B: D20 Maps

In [ ]:
# D20 depth maps for each experiment
cmap_d20, norm_d20, levels_d20 = climplot.sequential_cmap(0, 300, 20, cmap_name="viridis_r")

for group in diag.groups:
    r = results[group.name]

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5),
                             subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)})

    # Model D20
    ax = axes[0]
    ax.set_facecolor("#808080")
    ds_sub = r["ds_sub"]
    if r["geolon_c"] is not None:
        cb = ax.pcolormesh(r["geolon_c"], r["geolat_c"], r["d20_map"],
                           transform=ccrs.PlateCarree(), cmap=cmap_d20, norm=norm_d20)
    else:
        lon2d = ds_sub.geolon.values if "geolon" in ds_sub else None
        lat2d = ds_sub.geolat.values if "geolat" in ds_sub else None
        cb = ax.pcolormesh(lon2d, lat2d, r["d20_map"],
                           transform=ccrs.PlateCarree(), cmap=cmap_d20, norm=norm_d20)
    ax.coastlines(linewidth=0.5)
    ax.set_extent([100, 300, -TROP_BAND, TROP_BAND], crs=ccrs.PlateCarree())
    ax.set_title(f"{group.name}")

    # WOA13 D20
    ax = axes[1]
    obs_key = r["obs_key"]
    obs = obs_results[obs_key]
    dsobs_sub = obs["dsobs_sub"]
    ax.set_facecolor("#808080")
    lon2d_obs = dsobs_sub.geolon.values if "geolon" in dsobs_sub else None
    lat2d_obs = dsobs_sub.geolat.values if "geolat" in dsobs_sub else None
    cb = ax.pcolormesh(lon2d_obs, lat2d_obs, obs["d20_map"],
                       transform=ccrs.PlateCarree(), cmap=cmap_d20, norm=norm_d20)
    ax.coastlines(linewidth=0.5)
    ax.set_extent([100, 300, -TROP_BAND, TROP_BAND], crs=ccrs.PlateCarree())
    ax.set_title("WOA13")

    plt.subplots_adjust(bottom=0.15)
    cax = fig.add_axes([0.15, 0.05, 0.7, 0.025])
    fig.colorbar(cb, cax=cax, orientation="horizontal", extend="max",
                 label="D20 Depth (m)")
    fig.suptitle(f"20°C Isotherm Depth — {group.name} vs WOA13", y=1.01)
    all_figs.append(fig)

### Analysis C: Zonal D20 Profile Along the Equator

In [ ]:
# Equatorial D20 profile: the key tilt figure
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)

# Panel 1: D20 along equator
ax = axes[0]
for group in diag.groups:
    r = results[group.name]
    lon = r["geolon_1d"].values if hasattr(r["geolon_1d"], "values") else r["geolon_1d"]
    ax.plot(lon, r["d20_eq"], color=group.plot_color, linewidth=1.5,
            label=group.name)

# Plot WOA13 (use first obs_key found — they share the same equatorial band)
first_obs_key = list(obs_results.keys())[0]
obs = obs_results[first_obs_key]
lon_obs = obs["geolon_1d"].values if hasattr(obs["geolon_1d"], "values") else obs["geolon_1d"]
ax.plot(lon_obs, obs["d20_eq"], color="k", linewidth=2, linestyle="--",
        label="WOA13")

ax.invert_yaxis()
ax.set_ylabel("D20 Depth (m)")
ax.set_title(f"20°C Isotherm Depth Along Equator (±{EQ_BAND}°N/S avg)")
ax.legend(fontsize=8)
ax.grid(linewidth=0.5, alpha=0.5)

# Panel 2: Max dT/dz depth along equator
ax = axes[1]
for group in diag.groups:
    r = results[group.name]
    lon = r["geolon_1d"].values if hasattr(r["geolon_1d"], "values") else r["geolon_1d"]
    ax.plot(lon, r["d_maxgrad_eq"], color=group.plot_color, linewidth=1.5,
            label=group.name)

ax.plot(lon_obs, obs["d_maxgrad_eq"], color="k", linewidth=2, linestyle="--",
        label="WOA13")

ax.invert_yaxis()
ax.set_ylabel("Depth of max |dT/dz| (m)")
ax.set_xlabel("Longitude")
ax.set_title("Maximum Vertical Temperature Gradient Depth")
ax.legend(fontsize=8)
ax.grid(linewidth=0.5, alpha=0.5)

plt.tight_layout()
all_figs.append(fig)

### Analysis D: Tilt Scalar Metrics

In [ ]:
# Compute tilt metrics: D20 in western vs eastern boxes
tilt_data = {}

for group in diag.groups:
    r = results[group.name]
    lon = r["geolon_1d"].values if hasattr(r["geolon_1d"], "values") else np.asarray(r["geolon_1d"])

    # Western Pacific box (0-360 convention)
    west_mask = (lon >= WEST_BOX[0]) & (lon <= WEST_BOX[1])
    d20_west = np.nanmean(r["d20_eq"][west_mask]) if west_mask.any() else np.nan

    # Eastern Pacific box (0-360 convention)
    east_mask = (lon >= EAST_BOX[0]) & (lon <= EAST_BOX[1])
    d20_east = np.nanmean(r["d20_eq"][east_mask]) if east_mask.any() else np.nan

    tilt = d20_west - d20_east

    tilt_data[group.name] = {
        "d20_west": d20_west,
        "d20_east": d20_east,
        "tilt": tilt,
    }

    # Register metrics
    group.add_metric("d20", ("western_pacific_m", round(float(d20_west), 1)))
    group.add_metric("d20", ("eastern_pacific_m", round(float(d20_east), 1)))
    group.add_metric("tilt", ("magnitude_m", round(float(tilt), 1)))

# WOA13 tilt
for obs_key, obs in obs_results.items():
    lon_obs = obs["geolon_1d"].values if hasattr(obs["geolon_1d"], "values") else np.asarray(obs["geolon_1d"])
    west_mask_obs = (lon_obs >= WEST_BOX[0]) & (lon_obs <= WEST_BOX[1])
    east_mask_obs = (lon_obs >= EAST_BOX[0]) & (lon_obs <= EAST_BOX[1])
    d20_west_obs = np.nanmean(obs["d20_eq"][west_mask_obs]) if west_mask_obs.any() else np.nan
    d20_east_obs = np.nanmean(obs["d20_eq"][east_mask_obs]) if east_mask_obs.any() else np.nan
    tilt_obs = d20_west_obs - d20_east_obs
    tilt_data["WOA13"] = {
        "d20_west": d20_west_obs,
        "d20_east": d20_east_obs,
        "tilt": tilt_obs,
    }

# Tilt bias vs WOA13
for group in diag.groups:
    tilt_bias = tilt_data[group.name]["tilt"] - tilt_data["WOA13"]["tilt"]
    group.add_metric("tilt", ("bias_vs_woa13_m", round(float(tilt_bias), 1)))

# Bar chart of tilt magnitude
fig, ax = plt.subplots(figsize=(6, 4))
names = [g.name for g in diag.groups] + ["WOA13"]
colors = [g.plot_color for g in diag.groups] + ["black"]
tilts = [tilt_data[n]["tilt"] for n in names]

bars = ax.bar(names, tilts, color=colors, alpha=0.8, edgecolor="k")
ax.axhline(0, color="k", linewidth=0.5)
ax.set_ylabel("Thermocline Tilt (m)")
ax.set_title(f"D20 West ({WEST_BOX[0]:.0f}\u00b0E\u2013{WEST_BOX[1]:.0f}\u00b0E) minus "
             f"East ({EAST_BOX[0]:.0f}\u00b0E\u2013{EAST_BOX[1]:.0f}\u00b0E)")

# Annotate bars with values
for bar, val in zip(bars, tilts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f"{val:.0f} m", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
all_figs.append(fig)

# Print summary table
print(f"{'Experiment':<25s} {'D20 West (m)':>12s} {'D20 East (m)':>12s} {'Tilt (m)':>10s}")
print("-" * 62)
for name in names:
    d = tilt_data[name]
    print(f"{name:<25s} {d['d20_west']:>12.1f} {d['d20_east']:>12.1f} {d['tilt']:>10.1f}")

### Analysis E: Seasonal Cycle of Thermocline Tilt

In [ ]:
# Monthly climatology of D20 along the equator
from momlevel.util import annual_cycle

for group in diag.groups:
    r = results[group.name]
    ds_sub = r["ds_sub"]
    z_l = r["z_l"]
    lon = r["geolon_1d"].values if hasattr(r["geolon_1d"], "values") else np.asarray(r["geolon_1d"])

    # Equatorial subset of monthly thetao — compute climatology lazily
    thetao_eq_mon = ds_sub.thetao.isel(yh=r["eq_indices"]).mean("yh")

    # Compute monthly climatology: group by month, mean, then load
    thetao_clim_mon = thetao_eq_mon.groupby("time.month").mean("time").load()

    # Compute D20 for each month
    d20_seasonal = np.zeros((12, len(lon)))
    for m in range(12):
        d20_seasonal[m] = isotherm_depth(thetao_clim_mon.isel(month=m), z_l, iso_temp=ISO_TEMP)

    # Hovmoller: month vs longitude
    fig, ax = plt.subplots(figsize=(8, 4))
    lon_i = momgrid.util.infer_bounds(lon)
    month_i = np.arange(0.5, 13.5)
    cmap_hov, norm_hov, _ = climplot.sequential_cmap(0, 300, 20, cmap_name="viridis_r")
    cb = ax.pcolormesh(lon_i, month_i, d20_seasonal, cmap=cmap_hov, norm=norm_hov)
    ax.set_yticks(range(1, 13))
    ax.set_yticklabels(["J", "F", "M", "A", "M", "J",
                        "J", "A", "S", "O", "N", "D"])
    ax.set_ylabel("Month")
    ax.set_xlabel("Longitude")
    ax.set_title(f"Seasonal D20 (\u00b1{EQ_BAND}\u00b0N/S) \u2014 {group.name}")
    plt.subplots_adjust(bottom=0.18)
    cax = fig.add_axes([0.15, 0.05, 0.7, 0.025])
    fig.colorbar(cb, cax=cax, orientation="horizontal", extend="max",
                 label="D20 Depth (m)")
    all_figs.append(fig)

In [ ]:
# Seasonal tilt index: western minus eastern D20 by month
fig, ax = plt.subplots(figsize=(6, 4))

for group in diag.groups:
    r = results[group.name]
    ds_sub = r["ds_sub"]
    z_l = r["z_l"]
    lon = r["geolon_1d"].values if hasattr(r["geolon_1d"], "values") else np.asarray(r["geolon_1d"])

    thetao_eq_mon = ds_sub.thetao.isel(yh=r["eq_indices"]).mean("yh")
    thetao_clim_mon = thetao_eq_mon.groupby("time.month").mean("time").load()

    west_mask = (lon >= WEST_BOX[0]) & (lon <= WEST_BOX[1])
    east_mask = (lon >= EAST_BOX[0]) & (lon <= EAST_BOX[1])

    tilt_seasonal = np.zeros(12)
    for m in range(12):
        d20_mon = isotherm_depth(thetao_clim_mon.isel(month=m), z_l, iso_temp=ISO_TEMP)
        d20_w = np.nanmean(d20_mon[west_mask])
        d20_e = np.nanmean(d20_mon[east_mask])
        tilt_seasonal[m] = d20_w - d20_e

    ax.plot(range(1, 13), tilt_seasonal, color=group.plot_color,
            linewidth=1.5, marker="o", markersize=4, label=group.name)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J", "F", "M", "A", "M", "J",
                    "J", "A", "S", "O", "N", "D"])
ax.set_xlabel("Month")
ax.set_ylabel("Tilt (m)")
ax.set_title("Seasonal Cycle of Thermocline Tilt (D20 West \u2212 East)")
ax.legend(fontsize=8)
ax.grid(linewidth=0.5, alpha=0.5)
plt.tight_layout()
all_figs.append(fig)

### Analysis F: Time Evolution and Hovmoller

In [ ]:
# Annual-mean D20 along equator vs time (Hovmoller)
for group in diag.groups:
    r = results[group.name]
    lon = r["geolon_1d"].values if hasattr(r["geolon_1d"], "values") else r["geolon_1d"]
    d20_eq_ann = r["d20_eq_annual"].values

    fig, ax = plt.subplots(figsize=(8, 4))
    lon_i = momgrid.util.infer_bounds(lon)
    years = r["d20_eq_annual"].year.values
    year_i = momgrid.util.infer_bounds(years.astype(float))
    cmap_hov, norm_hov, _ = climplot.sequential_cmap(0, 300, 20, cmap_name="viridis_r")
    cb = ax.pcolormesh(lon_i, year_i, d20_eq_ann, cmap=cmap_hov, norm=norm_hov)
    ax.set_ylabel("Year")
    ax.set_xlabel("Longitude")
    ax.set_title(f"Annual D20 (±{EQ_BAND}°N/S) — {group.name}")
    plt.subplots_adjust(bottom=0.18)
    cax = fig.add_axes([0.15, 0.05, 0.7, 0.025])
    fig.colorbar(cb, cax=cax, orientation="horizontal", extend="max",
                 label="D20 Depth (m)")
    all_figs.append(fig)

In [ ]:
# Timeseries of thermocline tilt index (D20 west - east) over time
fig, ax = plt.subplots(figsize=(8, 4))

for group in diag.groups:
    r = results[group.name]
    lon = r["geolon_1d"].values if hasattr(r["geolon_1d"], "values") else np.asarray(r["geolon_1d"])
    d20_eq_ann = r["d20_eq_annual"].values
    years = r["d20_eq_annual"].year.values

    west_mask = (lon >= WEST_BOX[0]) & (lon <= WEST_BOX[1])
    east_mask = (lon >= EAST_BOX[0]) & (lon <= EAST_BOX[1])

    tilt_ts = np.nanmean(d20_eq_ann[:, west_mask], axis=1) - \
              np.nanmean(d20_eq_ann[:, east_mask], axis=1)

    ax.plot(years, tilt_ts, color=group.plot_color, linewidth=1.5,
            label=group.name)

    # Register interannual variability metric
    group.add_metric("tilt", ("std_m", round(float(np.nanstd(tilt_ts)), 1)))
    group.add_metric("tilt", ("mean_annual_m", round(float(np.nanmean(tilt_ts)), 1)))

# WOA13 reference line
if "WOA13" in tilt_data:
    ax.axhline(tilt_data["WOA13"]["tilt"], color="k", linewidth=1.5,
               linestyle="--", label="WOA13")

ax.set_xlabel("Year")
ax.set_ylabel("Tilt (m)")
ax.set_title("Interannual Thermocline Tilt (D20 West \u2212 East)")
ax.legend(fontsize=8)
ax.grid(linewidth=0.5, alpha=0.5)
plt.tight_layout()
all_figs.append(fig)

## Write Metrics

In [ ]:
diag.write_metrics("eq_pacific_thermocline_tilt_metrics.json")

## Save PowerPoint

In [ ]:
nbtools.save_pptx(all_figs, "eq_pacific_thermocline_tilt.pptx")